In [278]:
def get_co_rated(matrix, a, b, axis="index"):
    if axis == "index":
        vec_a = matrix.loc[a] #.loc[a] selects a row
        vec_b = matrix.loc[b]

    elif axis == "columns":
        vec_a = matrix[a]
        vec_b = matrix[b]

    # keep only positions where both entries are present
    mask = vec_a.notna() & vec_b.notna()  #notna return true if the value isn't missing 

    return (
        vec_a[mask].to_numpy(dtype=float),
        vec_b[mask].to_numpy(dtype=float)
    )

def euclidean_similarity(vec1, vec2):
    vec1 = np.asarray(vec1, dtype=float)
    vec2 = np.asarray(vec2, dtype=float)

    # Check for valid lengths and finite numbers
    if len(vec1) != len(vec2) or len(vec1) < 2:
        return np.nan
    
    if not (np.isfinite(vec1).all() and np.isfinite(vec2).all()):
        return np.nan

    # Calculate Euclidean distance and convert to similarity
    distance = np.sqrt(np.sum((vec1 - vec2) ** 2))
    return 1 / (1 + distance)
    
def cosine_similarity(vec1, vec2):
    vec1 = np.asarray(vec1, dtype=float)
    vec2 = np.asarray(vec2, dtype=float)

    # Check for valid lengths and finite numbers
    if len(vec1) != len(vec2) or len(vec1) < 2:
        return np.nan
    
    if not (np.isfinite(vec1).all() and np.isfinite(vec2).all()):
        return np.nan

    # Calculate Cosine similarity
    dot_product = np.dot(vec1, vec2)

    norm_1 = np.linalg.norm(vec1)
    norm_2 = np.linalg.norm(vec2)

    if norm_1 == 0 or norm_2 == 0:
        return np.nan


    return dot_product / (norm_1 * norm_2)
def pearson_similarity(vec1, vec2):
    vec1 = np.asarray(vec1, dtype=float)
    vec2 = np.asarray(vec2, dtype=float)

    if len(vec1) != len(vec2) or len(vec1) < 2:
        return np.nan

    if not np.isfinite(vec1).all() or not np.isfinite(vec2).all():
        return np.nan

    centered_1 = vec1 - np.mean(vec1)
    centered_2 = vec2 - np.mean(vec2)

    # Use the centered vectors, not the original vectors
    norm_1 = np.linalg.norm(centered_1)
    norm_2 = np.linalg.norm(centered_2)

    if norm_1 == 0 or norm_2 == 0:
        return np.nan

    return np.dot(centered_1, centered_2) / (norm_1 * norm_2)

In [279]:
def find_k_nearest_users(matrix, target_user, k, similarity_fn):
    users=matrix.index.tolist()
    k_nearest_users = {}
    for user in users:
        if user != target_user:
            user_a,user_b=get_co_rated(matrix, target_user, user, axis="index")
            if(len(user_a)>=2):
                similarity= float(similarity_fn(user_a, user_b))
                k_nearest_users[user] = similarity

    k_nearest_users = sorted(k_nearest_users.items(), key=lambda x: x[1], reverse=True)
    if len(k_nearest_users) < k:
        print("Only found "+ str(len(k_nearest_users)) + "nearest users")
        return k_nearest_users
    return k_nearest_users[:k]
    

In [280]:
def predict_rating_user_based(matrix, target_user, target_item, k, similarity_fn) :
    k_nearest_users = find_k_nearest_users(matrix, target_user, k, similarity_fn)
    if(len(k_nearest_users)<k):
        print(f"Warning: Found only {len(k_nearest_users)} neighbors for user {target_user}. Expected {k}.")
        return np.nan
    numerator = 0
    denominator = 0
    for user in k_nearest_users:
        user_id=user[0]
        rating = matrix.loc[user_id, target_item]
        if pd.notna(rating):
            similarity = user[1]
            numerator += similarity * rating
            denominator += abs(similarity)
    if denominator == 0:
        return np.nan
    return numerator/denominator



In [281]:
def find_k_nearest_items(matrix, target_item, k, similarity_fn):
    movies=matrix.columns.tolist()
    k_nearest_movies={}
    for movie in movies:
        if movie!=target_item:
            item_a,item_b=get_co_rated(matrix, target_item, movie, axis="columns")
            if(len(item_a)>=2):
                similarity= float(similarity_fn(item_a, item_b))
                k_nearest_movies[movie] = similarity
    k_nearest_movies = sorted(k_nearest_movies.items(), key=lambda x: x[1], reverse=True)
    if len(k_nearest_movies) < k:
        print("Only found "+ str(len(k_nearest_movies)) + "nearest movies")
        return k_nearest_movies
    return k_nearest_movies[:k]

In [282]:
def predict_rating_item_based(matrix, target_user, target_item, k, similarity_fn):
    k_nearest_items= find_k_nearest_items(matrix, target_item, k, similarity_fn)
    if len(k_nearest_items)<k:
        print(f"Warning: Only found {len(k_nearest_items)} neighbors for item {target_item}. Requested {k}.")
        return np.nan
    numerator=0
    denominator=0
    for item in k_nearest_items:
        item_id=item[0]
        rating=matrix.loc[target_user,item_id]
        if pd.notna(rating):
            similarity = item[1]
            numerator += similarity * rating
            denominator += abs(similarity)
    if denominator == 0:
        return np.nan
    return numerator/denominator



I reused both methods from part C, but instead of searching the rows(users), I searched the columns (movies) to get the nearest movies to the target movie, also the prediction was made on the nearest movies that the user rated.

In [283]:
def mean_centering(matrix,target_user,target_item,k,similarity_fn):
    users=matrix.index.tolist()
    centered_matrix = pd.DataFrame(np.nan,index=matrix.index,columns=matrix.columns)
    for user in users:
        ratings=np.asarray(matrix.loc[user])
        ratings_centered=ratings-np.nanmean(ratings)
        centered_matrix.loc[user]=ratings_centered
    print(centered_matrix)
    rating=predict_rating_user_based(centered_matrix,target_user,target_item,k,similarity_fn)
    mean_user_rating=np.nanmean(np.asarray(matrix.loc[target_user]))
    return rating+mean_user_rating
        

In [284]:
import pandas as pd
import numpy as np

# Create a small sample rating matrix
# Rows = Users (1, 2, 3), Columns = Movies (101, 102, 103, 104)
data = {
    101: [5.0, 3.0, np.nan],
    102: [4.0, 2.0, 1.0],
    103: [5.0, 3.0, 5.0],
    104: [3.0, np.nan, 2.0]
}

test_matrix = pd.DataFrame(data, index=[1, 2, 3])
test_matrix.index.name = 'userId'

print("Test Rating Matrix:")
print(test_matrix)
print("\nFinding K Nearest Users for User 1 (k=2) using Cosine Similarity:")
k_nearest_users = find_k_nearest_users(test_matrix, 1, 2,cosine_similarity)
print(k_nearest_users)
print ("\nPredicting rating for User 1 on Movie 104 using k=2 and Cosine Similarity:")
predicted_rating = predict_rating_user_based(test_matrix, 1, 104, 2, cosine_similarity)
print(predicted_rating)
k_nearest_items = find_k_nearest_items(test_matrix, 101, 2, cosine_similarity)
print("\nFinding K Nearest Items for Movie 101 (k=2) using Cosine Similarity:")
print(k_nearest_items)
print("The predicted rating based on other rated movies: ")
predicted_rating=predict_rating_item_based(test_matrix, 3, 101, 3, cosine_similarity)
print(predicted_rating)
print("The predicted rating centered: ")
print(mean_centering(test_matrix,3,101,2,cosine_similarity))
print("The predicted rating using pearson: ")
print(predict_rating_user_based(test_matrix,3,101,2,pearson_similarity))

Test Rating Matrix:
        101  102  103  104
userId                    
1       5.0  4.0  5.0  3.0
2       3.0  2.0  3.0  NaN
3       NaN  1.0  5.0  2.0

Finding K Nearest Users for User 1 (k=2) using Cosine Similarity:
[(2, 0.997241374054808), (3, 0.9036961141150639)]

Predicting rating for User 1 on Movie 104 using k=2 and Cosine Similarity:
2.0

Finding K Nearest Items for Movie 101 (k=2) using Cosine Similarity:
[(103, 1.0), (102, 0.9970544855015815)]
The predicted rating based on other rated movies: 
Only found 2nearest movies
nan
The predicted rating centered: 
             101       102       103       104
userId                                        
1       0.750000 -0.250000  0.750000 -1.250000
2       0.333333 -0.666667  0.333333       NaN
3            NaN -1.666667  2.333333 -0.666667
3.182531959116967
The predicted rating using pearson: 
3.837599039231231
